In [19]:
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output


class NPZReplayViewer:
    def __init__(self, npz_path):
        self.npz_path = Path(npz_path)
        self.data = np.load(self.npz_path, allow_pickle=True)

        self.positions = self.data["positions"]      # (T, N, 2)
        self.goals = self.data["goals"]              # (N, 2)
        self.obstacles = self.data["obstacles"]      # (H, W)

        self.num_steps = self.positions.shape[0]
        self.num_agents = self.positions.shape[1]
        self.num_rows, self.num_cols = self.obstacles.shape

        self.current_timestep = 0

        self._build_widgets()

    def _build_widgets(self):
        self.slider = widgets.IntSlider(
            value=0,
            min=0,
            max=self.num_steps - 1,
            step=1,
            description="Timestep:",
            continuous_update=False,
            layout=widgets.Layout(width="500px"),
        )

        self.prev_button = widgets.Button(description="Prev")
        self.next_button = widgets.Button(description="Next")

        self.slider.observe(self._on_slider_change, names="value")
        self.prev_button.on_click(self._on_prev_clicked)
        self.next_button.on_click(self._on_next_clicked)

        self.out = widgets.Output()

        self.controls = widgets.VBox([
            self.slider,
            widgets.HBox([self.prev_button, self.next_button]),
        ])

    def _render(self):
        with self.out:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(6, 6))

            # Draw grid
            for r in range(self.num_rows):
                for c in range(self.num_cols):
                    color = "black" if self.obstacles[r, c] else "white"
                    rect = patches.Rectangle(
                        (c, r),
                        1,
                        1,
                        facecolor=color,
                        edgecolor="gray",
                    )
                    ax.add_patch(rect)

            # Draw goals
            for i, (gr, gc) in enumerate(self.goals):
                ax.text(
                    gc + 0.5,
                    gr + 0.2,
                    f"G{i}",
                    ha="center",
                    va="center",
                    fontsize=8,
                )

            # Draw agents
            positions_t = self.positions[self.current_timestep]

            for i, (r, c) in enumerate(positions_t):
                circle = patches.Circle((c + 0.5, r + 0.5), 0.3)
                ax.add_patch(circle)

                ax.text(
                    c + 0.5,
                    r + 0.5,
                    str(i),
                    ha="center",
                    va="center",
                )

            ax.set_title(
                f"{self.npz_path.name} | "
                f"t={self.current_timestep}/{self.num_steps - 1}"
            )

            ax.set_xlim(0, self.num_cols)
            ax.set_ylim(self.num_rows, 0)
            ax.set_aspect("equal")
            ax.set_xticks(range(self.num_cols + 1))
            ax.set_yticks(range(self.num_rows + 1))
            ax.grid(True)

            plt.show()

    def _on_slider_change(self, change):
        self.current_timestep = change["new"]
        self._render()

    def _on_prev_clicked(self, _):
        if self.current_timestep > 0:
            self.slider.value = self.current_timestep - 1

    def _on_next_clicked(self, _):
        if self.current_timestep < self.num_steps - 1:
            self.slider.value = self.current_timestep + 1

    def show(self):
        display(self.controls)
        display(self.out)
        self._render()

In [21]:
viewer = NPZReplayViewer("dataset/held_out/ckpt_30000/random_ms147_ss1002_na48.npz")
viewer.show()

Output()